# Silver to Gold: fact_sales v2 (con dim_product y dim_province)

Prerequisitos: ejecutar primero silver_to_gold_dim_product y silver_to_gold_dim_province.

Star schema resultante:
- date -> dim_calendar.date
- customer_id -> dim_customer.customer_id
- delegate_id -> dim_delegate.delegate_id
- product_id -> dim_product.product_id (NUEVO, -1 si line/agrup son nulos)
- province_id -> dim_province.province_id

In [ ]:
%run ./config

SILVER_TABLE       = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
GOLD_TABLE         = f"{DEFAULT_SCHEMA}.fact_sales"
GOLD_DIM_CUSTOMER  = f"{DEFAULT_SCHEMA}.dim_customer"
GOLD_DIM_CALENDAR  = f"{DEFAULT_SCHEMA}.dim_calendar"
GOLD_DIM_DELEGATE  = f"{DEFAULT_SCHEMA}.dim_delegate"
GOLD_DIM_PRODUCT   = f"{DEFAULT_SCHEMA}.dim_product"
GOLD_DIM_PROVINCE  = f"{DEFAULT_SCHEMA}.dim_province"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")
print(f"Filas leidas desde Silver: {df_silver.count()}")
df_silver.printSchema()

In [ ]:
df_dim_customer = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_CUSTOMER}")
df_dim_calendar = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_CALENDAR}")
df_dim_delegate = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_DELEGATE}")
df_dim_product  = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_PRODUCT}")
df_dim_province = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_PROVINCE}")

print(f"dim_product filas  : {df_dim_product.count()}")
print(f"dim_province filas : {df_dim_province.count()}")

In [ ]:
COLS_TO_DROP = ["customer_name", "delegate_name", "province_name", "_silver_load_ts"]
cols_to_drop = [c for c in COLS_TO_DROP if c in df_silver.columns]

# Normalizar las columnas de producto igual que en dim_product
df_base = (
    df_silver
    .drop(*cols_to_drop)
    .withColumn("_line_norm",   F.trim(F.initcap(F.col("line"))))
    .withColumn("_agrup1_norm", F.trim(F.initcap(F.col("agrup1"))))
    .withColumn("_agrup2_norm", F.trim(F.initcap(F.col("agrup2"))))
)

# Join LEFT para obtener product_id (null si no hay match)
df_prod_lookup = df_dim_product.select(
    "product_id",
    F.col("line").alias("_p_line"),
    F.col("agrup1").alias("_p_agrup1"),
    F.col("agrup2").alias("_p_agrup2")
)

df_base = (
    df_base
    .join(
        df_prod_lookup,
        (df_base["_line_norm"]   == df_prod_lookup["_p_line"]) &
        (df_base["_agrup1_norm"] == df_prod_lookup["_p_agrup1"]) &
        (df_base["_agrup2_norm"] == df_prod_lookup["_p_agrup2"]),
        how="left"
    )
    .drop("_line_norm", "_agrup1_norm", "_agrup2_norm",
          "_p_line", "_p_agrup1", "_p_agrup2")
)

sin_producto = df_base.filter(F.col("product_id").isNull()).count()
print(f"Filas sin product_id (line/agrup nulos en Silver): {sin_producto}")

# Asignar -1 (producto desconocido) para no perder filas
df_base = df_base.withColumn(
    "product_id",
    F.when(F.col("product_id").isNull(), F.lit(-1).cast("integer"))
     .otherwise(F.col("product_id"))
)

In [ ]:
rows_before = df_base.count()

df_gold = (
    df_base
    .join(df_dim_customer.select("customer_id"), on="customer_id", how="inner")
    .join(df_dim_calendar.select("date"),        on="date",        how="inner")
    .join(df_dim_delegate.select("delegate_id"), on="delegate_id", how="inner")
    .join(df_dim_province.select("province_id"), on="province_id", how="inner")
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

rows_after = df_gold.count()
print(f"Filas antes del filtro referencial : {rows_before}")
print(f"Filas despues del filtro           : {rows_after}")
print(f"Filas descartadas (huerfanas)      : {rows_before - rows_after}")

In [ ]:
row_count      = df_gold.count()
null_products  = df_gold.filter(F.col("product_id").isNull()).count()
null_provinces = df_gold.filter(F.col("province_id").isNull()).count()

print(f"Filas a escribir en Gold   : {row_count}")
print(f"product_id nulos           : {null_products}")
print(f"province_id nulos          : {null_provinces}")

assert null_products  == 0, "ERROR: product_id nulos tras asignacion de -1"
assert null_provinces == 0, "ERROR: province_id nulos en fact_sales Gold"

df_gold.show(3)

In [ ]:
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"Tabla {GOLD_TABLE} escrita en {GOLD_LAKEHOUSE} con {row_count} filas.")

# Silver → Gold: fact_sales (v2 — con dim_product y dim_province)

**Propósito:** Actualizar `fact_sales` añadiendo las FKs a las nuevas dimensiones:
- `product_id` → FK a `dim_product` (join por agrup1 + agrup2 + line)
- `province_id` ya existe como campo nativo — se valida contra `dim_province`

**Prerequisitos:** Ejecutar primero:
1. `silver_to_gold_dim_product`
2. `silver_to_gold_dim_province`

**Star schema resultante:**
```
fact_sales
  ├── date          → dim_calendar.date
  ├── customer_id   → dim_customer.customer_id
  ├── delegate_id   → dim_delegate.delegate_id
  ├── product_id    → dim_product.product_id  ← NUEVO
  └── province_id   → dim_province.province_id ← validado
```

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [ ]:
%run ./config

SILVER_TABLE        = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
GOLD_TABLE          = f"{DEFAULT_SCHEMA}.fact_sales"

GOLD_DIM_CUSTOMER   = f"{DEFAULT_SCHEMA}.dim_customer"
GOLD_DIM_CALENDAR   = f"{DEFAULT_SCHEMA}.dim_calendar"
GOLD_DIM_DELEGATE   = f"{DEFAULT_SCHEMA}.dim_delegate"
GOLD_DIM_PRODUCT    = f"{DEFAULT_SCHEMA}.dim_product"
GOLD_DIM_PROVINCE   = f"{DEFAULT_SCHEMA}.dim_province"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas leídas desde Silver: {df_silver.count()}")
df_silver.printSchema()

In [ ]:
# ─── CARGAR DIMENSIONES ───────────────────────────────────────────────────────
df_dim_customer  = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_CUSTOMER}")
df_dim_calendar  = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_CALENDAR}")
df_dim_delegate  = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_DELEGATE}")
df_dim_product   = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_PRODUCT}")
df_dim_province  = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_PROVINCE}")

print(f"dim_product filas  : {df_dim_product.count()}")
print(f"dim_province filas : {df_dim_province.count()}")

In [ ]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────
COLS_TO_DROP = [
    "customer_name",
    "delegate_name",
    "province_name",   # ya en dim_province
    "_silver_load_ts",
]

existing_cols = df_silver.columns
cols_to_drop  = [c for c in COLS_TO_DROP if c in existing_cols]

# Normalizar agrup1/agrup2/line antes del join (igual que en dim_product)
df_base = (
    df_silver
    .drop(*cols_to_drop)
    .withColumn("_line_norm",   F.trim(F.initcap(F.col("line"))))
    .withColumn("_agrup1_norm", F.trim(F.initcap(F.col("agrup1"))))
    .withColumn("_agrup2_norm", F.trim(F.initcap(F.col("agrup2"))))
)

# Join para obtener product_id
df_dim_product_join = df_dim_product.select(
    "product_id",
    F.col("line").alias("_p_line"),
    F.col("agrup1").alias("_p_agrup1"),
    F.col("agrup2").alias("_p_agrup2")
)

df_base = (
    df_base
    .join(
        df_dim_product_join,
        (df_base["_line_norm"]   == df_dim_product_join["_p_line"]) &
        (df_base["_agrup1_norm"] == df_dim_product_join["_p_agrup1"]) &
        (df_base["_agrup2_norm"] == df_dim_product_join["_p_agrup2"]),
        how="left"
    )
    .drop("_line_norm", "_agrup1_norm", "_agrup2_norm",
          "_p_line", "_p_agrup1", "_p_agrup2")
)

print(f"Filas tras join con dim_product: {df_base.count()}")
print(f"Filas con product_id nulo      : {df_base.filter(F.col('product_id').isNull()).count()}")

In [ ]:
# ─── FILTRO REFERENCIAL ───────────────────────────────────────────────────────
rows_before = df_base.count()

df_gold = (
    df_base
    .join(df_dim_customer.select("customer_id"), on="customer_id", how="inner")
    .join(df_dim_calendar.select("date"),        on="date",        how="inner")
    .join(df_dim_delegate.select("delegate_id"), on="delegate_id", how="inner")
    .join(df_dim_province.select("province_id"), on="province_id", how="inner")
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

rows_after = df_gold.count()
print(f"Filas antes del filtro referencial : {rows_before}")
print(f"Filas después del filtro           : {rows_after}")
print(f"Filas descartadas (huérfanas)      : {rows_before - rows_after}")

In [ ]:
# ─── VALIDACIÓN ───────────────────────────────────────────────────────────────
row_count      = df_gold.count()
null_products  = df_gold.filter(F.col("product_id").isNull()).count()
null_provinces = df_gold.filter(F.col("province_id").isNull()).count()

print(f"Filas a escribir en Gold   : {row_count}")
print(f"product_id nulos           : {null_products}")
print(f"province_id nulos          : {null_provinces}")

assert null_products  == 0, "ERROR: hay product_id nulos en fact_sales Gold"
assert null_provinces == 0, "ERROR: hay province_id nulos en fact_sales Gold"

df_gold.show(3)

In [ ]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"Tabla {GOLD_TABLE} escrita en {GOLD_LAKEHOUSE} con {row_count} filas.")